# BME 590 - Exercise 0 - Setup & Introduction
**Professor:** Emma Chory, Ph.D.

**Authors:** 
Rick Wierenga, Joe Laforet, Stefan Golas, Ben Perry

---

### Welcome to PyLabRobot!
PyLabRobot *(PLR)* is a universal Python hardware and operating system-agnostic software development kit for automated and autonomous laboratories. PyLabRobot enables control of liquid handling robots, plate readers, pumps, scales, heater shakers, and other equiprment by converting Pythong commands to their corresponding low-level firmware/IO commands. The primary piece of equipment we care about controlling is the liquid handler, which is a robot that can aspirate and dispense precise volumes of liquid in a Cartesian coordinate system, essentially the same as hand pipetting, but automated!

PLR defines a universal interface class called **LiquidHandler** that provides generic methods for controlling liquid handlers (and some other classes, such as **PlateReader** for controlling other equipment like a plate reader). These **interface classes** are able to translate singular (**atomic**) commands for a robot (aspirate, drop tips, dispense, move, etc.) to any number of **supported robots** via custom **backends** (or drivers). \[See Figure 1 for a diagram of how this works\] This setup enables the definition of a protocol in Python, subsequent translation to any number of robots, all in one Python script or notebook!


<div>
<img src="../figs/fig_1_plr.jpg" width="1000"/>
</div>


As we work through exercises and tutorials for PLR, make note of these specific help resources! Specifically, reading the **publication below** will help you get oriented with the organization of PLR. For more specific questions, reach out on the class **slack** or to the **TA** or **Professor** via email.

**GitHub:** [Link](https://github.com/PyLabRobot/pylabrobot)

**PLR Forum:** [Link](https://discuss.pylabrobot.org/)

**Publication:** [Link](https://docs.pylabrobot.org/index.html#:~:text=Wierenga%20et%20al%202023%2C%20PyLabRobot%3A%20An%20Open%2DSource%2C%20Hardware%20Agnostic%20Interface%20for%20Liquid%2DHandling%20Robots%20and%20Accessories%20(Device))

### Getting Started
#### PLR Installation
To get started with PyLabRobot in a research environment, you should follow the [installation instructions](https://docs.pylabrobot.org/user_guide/_getting-started/installation.html) in the online documentation, However, for BME 590, we provide a more robust set of installation instructions to ensure the entire class is on the same page to start. These instructions can be found on the class repository located [here](https://github.com/chory-lab/bme590-fall-2025). Technically, if you are seeing this notebook, you have at least located the repo so congrats!

Let's go ahead and test your PLR installation! You should have an output that appears like:

```txt
Opentrons Version:  8.5.1
PLR Version:  0.1.6
```

In [5]:
import pylabrobot
import opentrons

print("Opentrons Version: ", opentrons.__version__)
print("PLR Version: ", pylabrobot.__version__)

Opentrons Version:  8.5.1
PLR Version:  0.1.6


#### Auto-complete / Pylance Missing Imports Issue

Depending on your installation, you may get an issue where `import pylabrobot` runs, but VS Code thinks the import is missing from your provided environment. This can happen when the installation path of libraries is **disjoint**, meaning that VS Code normally will look in your **conda** environment for all the packages being used in this notebook. However, if **pylabrobot** was installed from source, it may be placed in another folder on the **base** environment, through pip, or somewhere else.

<div>
<img src="../figs/plr_pylance_error_fig.png" width="1000"/>
</div>

If this issue occurs on your install, the easiest way to fix it is to simply **locate the directory containing the PLR package** and **tell VS Code wehre it is**. To find the containing directory, run the following code and copy the output. Then, using the VS Code command palette (Ctrl/Cmd + P), search for **Preferences: Open User Settings (JSON)** and paste the following JSON output into that file. 

**Note -** If you already have settings present in the **settings.json** file, then just paste in the key-value pair for `"python.analysis.extraPaths"`

Finally, **close this file** and **reload VS Code** using the VS Code command palette (Ctrl/Cmd + P), searching for **Developer: Reload Window**

In [6]:
import os
import json
import pylabrobot
plr_path = pylabrobot.__file__
final_path = os.path.dirname(os.path.dirname(plr_path))
settings_dict = {
  "python.analysis.extraPaths": [
    final_path
  ]
}
print("--- Copy and paste this JSON below into your settings.json ---\n")
print(json.dumps(settings_dict, indent=4))

--- Copy and paste this JSON below into your settings.json ---

{
    "python.analysis.extraPaths": [
        "/Users/bcp28/Documents/duke/ta/emma_590/bme590-fall-2025/pylabrobot"
    ]
}


#### Setting Up Your First Deck & Visualizer

Let's go ahead and set up your first deck for a liquid handler. To set up a liquid handler, you will need three things:

- **Liquid Handler Interface** - This is the top level class that will organize everything else for us.
- **A Deck Layout** - Different machines have different deck sizes/slots. The deck layout tells the LiquidHandler the geometric constraints of our robot.
- **A Backend** -- The backend does the heavy lifting of converting the instructions we write in Python to **machine code** tailored to each robot. Because we do not have a lab component to this class (yet), we must do our experimentation *virtually*. Fortunately, PLR has a built-in `Visualizer` class, which enables us to host a website locally that will display our active experiment as we work!

Let's start with the `LiquidHandler` class. Run the following code.

In [7]:
from pylabrobot.liquid_handling import LiquidHandler

lh = LiquidHandler() # run this - you will get an error. This is ok!

TypeError: LiquidHandler.__init__() missing 2 required positional arguments: 'backend' and 'deck'

You should have gotten an **error** that looks like the following:

<div>
<img src="../figs/lh_error.png" width="800"/>
</div>

As expected, the `LiquidHandler` won't work without a suitable backend or deck! Let's try using a [Hamilton STAR](https://www.hamiltoncompany.com/microlab-star?srsltid=AfmBOoqFyaWUKXUD4qwBmI0-aVXG-9hHUi02XsAF6NUdgpA8M3uo9SSn) liquid handler. The deck layout should look like this!

<div>
<img src="../figs/star_deck.jpg" width="1100"/>
</div>

Normally, to use this deck, we would need the `STARBackend` to convert our commands to actual robot movements. Since we don't have a STAR machine to use for this class, we are going to use the `LiquidHandlerChatterboxBackend` instead, which will print pseudo-actions rather than moving an actual piece of lab machinery. Let's set up the deck now!

In [8]:
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import LiquidHandlerChatterboxBackend # import backend 
from pylabrobot.resources.hamilton import STARLetDeck # import deck layout from pylabrobot.resources.<manufacturer_name>

lh = LiquidHandler(backend = LiquidHandlerChatterboxBackend(),
                   deck = STARLetDeck())

Great! Now we have a deck with a backend. Finally, we need to call `setup()` to assign resources to the liquid handler. PyLabRobot uses a **directed, rooted tree** data structure to organize a liquid handler. In essence what this means is every `Resource` will have a `parent` and `children`, except the root. This means a deck can have children (such as a plate holder or a trash bin), and its children can have children (a plate to a plate holder), and so on (the children of a plate are wells within the plate.) 

This is best visualized in the figure below from the PLR paper, where the relationship between a plate well, plate, plate holder, and deck is viauzlied!

<div>
<img src="../figs/plr_tree_structure.jpeg" width="1100"/>
</div>

Calling `lh.setup()` will initialilze this data structure with the layout of the STARLet deck and pre-configure it with a trash bin!

In [9]:
try:
    await lh.setup()
except: # if lh is already running (i.e. this cell is run out of order) then reset via lh.stop()
    await lh.stop()
    await lh.setup()

Setting up the liquid handler.
Resource deck was assigned to the liquid handler.
Resource trash was assigned to the liquid handler.
Resource trash_core96 was assigned to the liquid handler.
Resource waste_block was assigned to the liquid handler.


Great! You should see an output akin to the following.

```txt
Setting up the liquid handler.
Resource deck was assigned to the liquid handler.
Resource trash was assigned to the liquid handler.
Resource trash_core96 was assigned to the liquid handler.
Resource waste_block was assigned to the liquid handler.
```

You can see the default setup for the STARLet deck will assign two trash bins and a waste block. These are children of the deck, which itself is a child of the liquid handler.

We can actually see the children of the deck by iterating through the `children` attribute of the deck.

In [10]:
for deck_child in lh.deck.children:
    print(f"Name: {deck_child.name}\nLocation: {deck_child.location}\n\n")

Name: trash
Location: Coordinate(800.000, 190.600, 137.100)


Name: trash_core96
Location: Coordinate(-58.200, 106.000, 229.000)


Name: waste_block
Location: Coordinate(775.000, 115.000, 100.000)




The default deck also will assign a `teaching_tip_rack` as a child of the `waste_block`. If you want to look at a specific resource itself, you can access it by calling `get_resource("<resource_name>")`. Let's look at the tip rack itself, which is a child of the waste_block.

In [11]:
# get_resource will find the waste block if it is a child of lh.deck
waste_block = lh.deck.get_resource("waste_block")
print(waste_block)

# now let's go ahead and get the teaching_tip_rack, which is a child of waste_block
teaching_rack = waste_block.get_resource("teaching_tip_rack")
print(teaching_rack)

# the teaching rack itself has children, which are the individual 8 wells!
for well in teaching_rack.children:
    print(f"Name: {well.name}\nLocation: {well.location}\n\n")

Resource(name=waste_block, location=Coordinate(775.000, 115.000, 100.000), size_x=30, size_y=445.2, size_z=100, category=None)
TipRack(name=teaching_tip_rack, size_x=9, size_y=72, size_z=50.4, location=Coordinate(005.900, 346.100, 000.000))
Name: teaching_tip_rack_tip_spot_0
Location: Coordinate(000.000, 063.000, 023.100)


Name: teaching_tip_rack_tip_spot_1
Location: Coordinate(000.000, 054.000, 023.100)


Name: teaching_tip_rack_tip_spot_2
Location: Coordinate(000.000, 045.000, 023.100)


Name: teaching_tip_rack_tip_spot_3
Location: Coordinate(000.000, 036.000, 023.100)


Name: teaching_tip_rack_tip_spot_4
Location: Coordinate(000.000, 027.000, 023.100)


Name: teaching_tip_rack_tip_spot_5
Location: Coordinate(000.000, 018.000, 023.100)


Name: teaching_tip_rack_tip_spot_6
Location: Coordinate(000.000, 009.000, 023.100)


Name: teaching_tip_rack_tip_spot_7
Location: Coordinate(000.000, 000.000, 023.100)




You can also directly access the teaching tip rack by using `get_resource` on the deck. This function will **recursively search all subtrees** to find the resource in question, if it exists.

In [12]:
# all the following are the same way to get the teaching_tip_rack.
method_1 = lh.get_resource("teaching_tip_rack") # search on root node (liquid handler)
method_2 = lh.deck.get_resource("teaching_tip_rack") # search on deck node (child of liquid handler)
method_3 = lh.deck.get_resource("waste_block").get_resource("teaching_tip_rack") # search on waste_block node (child of deck) 

print(method_1 == method_2 == method_3)

True


You can also print the total summary of the deck with the `lh.deck.summary()` command. Now we can actually see a printed summary of the deck, along with the same tree structure we were iterating over previously!

In [13]:
print(lh.deck.summary())

Rail  Resource                      Type           Coordinates (mm)
(-6)  ├── trash_core96              Trash          (-58.200, 106.000, 229.000)
      │
(31)  ├── waste_block               Resource       (775.000, 115.000, 100.000)
      │   ├── teaching_tip_rack     TipRack        (780.900, 461.100, 100.000)
      │
(32)  ├── trash                     Trash          (800.000, 190.600, 137.100)



Congrats! You have set up your first deck! This summary print statement is pretty convenient; hwoever, it is still hard to imagine in our head what is going on physically. To better visualize this, we will utilize the built in `Visualizer` class to host a website with an animation of what we are coding.

The `Visualizer` class expects a `resource` to be passed to its constructor as the resource to visualize. This should be the `LiquidHandler` class we defined above. Let's give it a try.

In [14]:
from pylabrobot.visualizer.visualizer import Visualizer

vis = Visualizer(resource=lh)
await vis.setup()

Websocket server started at http://127.0.0.1:2121
File server started at http://127.0.0.1:1337 . Open this URL in your browser.


Running the above code should output something similar to:

```txt
Websocket server started at http://127.0.0.1:XXXX
File server started at http://127.0.0.1:XXXX . Open this URL in your browser.
```

This is effectively hosting a website at 127.0.0.1, which is the **localhost**, a special IP address that means the server is being hosted on your own machine. This means when you open the URL provided in the output, you are **connecting** to your own machine's server. You should see a chrome (or whatever default browser your machine has) tab open with an output that is similar to this:

<div>
<img src="../figs/vis_ss.png" width="750"/>
</div>

If a browser doesn't automatically open, simply **copy/paste** the output URL to your browser of choice. 

The <span style="color:green"><strong>connected</strong></span> text in the top right corner of the visualizer indicates you are actively communicating with your local session. If for some reason this <span style="color:red"><strong>disconnects</strong></span>, the visualizer will need to be reset as it will lose track of any changes.

Congrats! Pylabrobot is working as expected! Let's add some actual labware now!

#### Assigning Labware

Remember the **tree** structure we were referring to earlier? Adding labware (such as tips, tubes, plates, etc.) is implemented by adding a **child node** to a parent. In most cases, you will simply be adding a **child resource** like a plate to the **deck**. 

Unless adding custom labware, almost all the resources you could possibly add are found in the `pylabrobot.resources` submodule. If you are looking for a specific plate type or resouce, simply search for it in this folder of source. 

If you are not familiar with `grep`, the next best way to search this is the GitHub search bar online or the search function in VS Code (More on this later. For this assignment, we will list out the relevant resources).

Let's go ahead and import some resources for the HamiltonStar

In [15]:
# get a tip carrier that holds 5 tip racks of 96 tips (480 total tips) [A00 revision]
from pylabrobot.resources import TIP_CAR_480_A00

# get a plate carrier that can hold 5 deep well 96 Well PCR Plates
from pylabrobot.resources import PLT_CAR_L5AC_A00

# get a corning 360 uL 96 well plate
from pylabrobot.resources import Cor_96_wellplate_360ul_Fb

# get a tip rack with 96 1000ul high volume tip with filter
from pylabrobot.resources import HTF

Great! Now we actually need to build our deck layout. First, let's go ahead and define a function that we can run that will input a deck, a backend, and a liquid handler object and create our visualizer!

In [16]:
from pylabrobot.resources import Deck
from pylabrobot.liquid_handling.backends.backend import LiquidHandlerBackend

async def visualize_deck(deck: Deck,
                         backend: LiquidHandlerBackend):
    try:
        lh = LiquidHandler(backend=backend, deck=deck)
        vis = Visualizer(resource = lh)
        await lh.setup()
        await vis.setup()
        return lh
    except Exception as e:
        print(f"Error! Got excpetion: {e}")

Now we need to actually create the object that will represent the tip carrier that we can assign to the deck. We need to give each object a unique name as that is the identifier we will use for everything downstream, such as assigning and unassigning resoruces, adding liquid, etc.

We will move these inside a function which sets up an entire deck in one-shot. The reason for this is once a resource is assigned, it gets a `parent` attribute. If cells are run out of order and the deck setup is not functionalized, an error will be thrown.

In general, **set up your deck completely inside a function** to save yourself headaches.

In [17]:
# create tip carrier object
tip_carrier = TIP_CAR_480_A00(name="awesome tip carrier 96x5")

# create plate carrier object
plate_carrier = PLT_CAR_L5AC_A00(name = "awesome plate carrier")

Now we need to **assign** this resource to the deck. This is akin to adding a child node to the deck node.

To do this, we will call the `assign_child_resource` method on the `STARLetDeck` object to assign the tip and plate carriers to specific locations. For the `STAR` deck layouts, the argument `rails` explains which set of rails to assign the plate carrier to. The way resources are assigned may be **different** for different deck layouts.

For example, the **OpenTrons2** does **not** have rails, but rather **delineated slots**. So if using an OT2, the method call will instead be `assign_child_at_slot` rather than `assign_child_resource`. 

In [18]:
async def make_deck_with_carriers():
    deck = STARLetDeck()
    tip_carrier = TIP_CAR_480_A00(name="awesome tip carrier 96x5")
    plate_carrier = PLT_CAR_L5AC_A00(name = "awesome plate carrier")
    deck.assign_child_resource(plate_carrier, rails = 5)
    deck.assign_child_resource(tip_carrier, rails = 11)
    return deck

deck = await make_deck_with_carriers()
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

Setting up the liquid handler.
Resource deck was assigned to the liquid handler.
Resource trash was assigned to the liquid handler.
Resource trash_core96 was assigned to the liquid handler.
Resource waste_block was assigned to the liquid handler.
Resource awesome plate carrier was assigned to the liquid handler.
Resource awesome tip carrier 96x5 was assigned to the liquid handler.
Websocket server started at http://127.0.0.1:2122
File server started at http://127.0.0.1:1338 . Open this URL in your browser.


Great! You should have a layout that looks like this on your visualizer!

<div>
<img src="../figs/carrier_layout.png" width="1200"/>
</div>

**Note** how the tip carriers are 5 rails wide and are assigned at the correct positions of **rail 5** and **rail 11**. Let's see what happens if we try to run the same code but place the carriers in illegal positions.

In [19]:
async def make_deck_with_carriers():
    deck = STARLetDeck()
    tip_carrier = TIP_CAR_480_A00(name="awesome tip carrier 96x5")
    plate_carrier = PLT_CAR_L5AC_A00(name = "awesome plate carrier")
    deck.assign_child_resource(plate_carrier, rails = 5)
    deck.assign_child_resource(tip_carrier, rails = 8)
    return deck

deck = await make_deck_with_carriers()
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

ValueError: Location Coordinate(257.500, 063.000, 100.000) is already occupied by resource 'awesome plate carrier'.

You should an error noting that the position is already occupied. Therefore, PLR will keep track of the current labware on your particular deck setup!.

```txt
ValueError: Location Coordinate(257.500, 063.000, 100.000) is already occupied by resource 'awesome plate carrier'.
```

You may have also noted we forgot to add tip racks and plates to our deck! Let's go ahead and add **2 tip racks** to our tip rack carrier and **4 plates** to our plate carrier. For **plate carriers**, you can index into their slots using python!

In [20]:
async def make_deck_with_carriers_and_contents():
    deck = STARLetDeck()

    # create carriers
    tip_carrier = TIP_CAR_480_A00(name="awesome tip carrier 96x5")
    plate_carrier = PLT_CAR_L5AC_A00(name = "awesome plate carrier")

    # assign carriers
    deck.assign_child_resource(plate_carrier, rails = 5)
    deck.assign_child_resource(tip_carrier, rails = 11)

    # define 2 tip racks with unique names and assign to the first two slots of the tip carrier
    for i in range(2):
        tip_carrier[i] = HTF(name=f"tip_rack_{i}")

    # define 4 plates with unique names and assign to the first two slots of the plate carrier
    for i in range(4):
        plate_carrier[i] = Cor_96_wellplate_360ul_Fb(name=f"plate_{i}")
        
    return deck

deck = await make_deck_with_carriers_and_contents()
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

Setting up the liquid handler.
Resource deck was assigned to the liquid handler.
Resource trash was assigned to the liquid handler.
Resource trash_core96 was assigned to the liquid handler.
Resource waste_block was assigned to the liquid handler.
Resource awesome plate carrier was assigned to the liquid handler.
Resource awesome tip carrier 96x5 was assigned to the liquid handler.
Websocket server started at http://127.0.0.1:2123
File server started at http://127.0.0.1:1339 . Open this URL in your browser.


Now we should have something looking like this!

<div>
<img src="../figs/star_layout.png" width="1200"/>
</div>

In the visualizer, you can actually hover your mouse over all components on the deck to see their names and how your layout is looking. Pretty neat! Let's see what the deck summary is looking like at this point.

In [21]:
print(lh.deck.summary())

Rail  Resource                        Type           Coordinates (mm)
(-6)  ├── trash_core96                Trash          (-58.200, 106.000, 229.000)
      │
(5)   ├── awesome plate carrier       PlateCarrier   (190.000, 063.000, 100.000)
      │   ├── plate_0                 Plate          (194.000, 071.500, 183.120)
      │   ├── plate_1                 Plate          (194.000, 167.500, 183.120)
      │   ├── plate_2                 Plate          (194.000, 263.500, 183.120)
      │   ├── plate_3                 Plate          (194.000, 359.500, 183.120)
      │   ├── <empty>
      │
(11)  ├── awesome tip carrier 96x5    TipCarrier     (325.000, 063.000, 100.000)
      │   ├── tip_rack_0              TipRack        (331.200, 073.000, 214.950)
      │   ├── tip_rack_1              TipRack        (331.200, 169.000, 214.950)
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │
(31)  ├── waste_block                 Resource       (775.000, 115.000, 100.000)
      │ 

#### Using Different Decks

When you are working with different decks, there are three primary changes that need to be considered in your code.

1. **Backend** - In practical application, you will need to change your backend to the correct one to match your hardware. For this class, we ignore this as we are using the simulator backend.
2. **Labware** - Labware from different companies will be compatible with different liquid handlers. You need to ensure the equipment you are importing is compatible with the machine you are using. In PLR, labware is organized by manufacturer, and using an incorrect labware will often throw an error if it is physically incompatible.
3. **Methods** - Not only will the layout of your liquid handler change, but sometimes the methods in Python to assign child resources change as well. For example, an **OpenTrons2** liquid handler has **slots** rather than **rails** in which it can assign lab equipment. Not only does the `assign` method change, but also the deck setup, as plate/tip carriers are not needed on an **OpenTrons2**.

In the following section, we will demonstrate setting up a similar deck layout to the STARLet deck, but on an OpenTrons2. Namely, we will need the following equipment on our deck:
- A trash bin
- 2 tip racks with 96 1000 uL tips
- 4 360 uL 96-well plates

Let's start again by importing the equipment we need. You can condense your import code a bit by providing the imports as a tuple.

In [22]:
from pylabrobot.resources import (
   opentrons_96_tiprack_1000ul, # 96 1000 uL tips
   corning_96_wellplate_360ul_flat # Corning 96-well 360 uL plate
)
from pylabrobot.resources.opentrons import OTDeck # need to import new OpenTrons Deck Layout

async def make_empty_ot2():
    deck = OTDeck()
    return deck

deck = await make_empty_ot2()
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

Setting up the liquid handler.
Resource deck was assigned to the liquid handler.
Resource trash_container was assigned to the liquid handler.
Websocket server started at http://127.0.0.1:2124
File server started at http://127.0.0.1:1340 . Open this URL in your browser.


You should see a deck with **11 slots** that looks like this:

<div>
<img src="../figs/ot_deck.png" width="600"/>
</div>

Now, lets's assign **tips** to slots 10 and 11 and **plates** to the bottom-left 4 slots.

In [23]:
async def make_same_ot2():
    deck = OTDeck()
    tip_rack_slots = [10, 11]
    plate_slots = [1, 2, 4, 5]
    for i, tip_rack_slot in enumerate(tip_rack_slots):
        deck.assign_child_at_slot(opentrons_96_tiprack_1000ul(name = f"tip_rack_{i}"), tip_rack_slot)
    for i, plate_slot in enumerate(plate_slots):
        deck.assign_child_at_slot(corning_96_wellplate_360ul_flat(name = f"plate_{i}"), plate_slot)
    return deck

deck = await make_same_ot2()
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

Setting up the liquid handler.
Resource deck was assigned to the liquid handler.
Resource trash_container was assigned to the liquid handler.
Resource tip_rack_0 was assigned to the liquid handler.
Resource tip_rack_1 was assigned to the liquid handler.
Resource plate_0 was assigned to the liquid handler.
Resource plate_1 was assigned to the liquid handler.
Resource plate_2 was assigned to the liquid handler.
Resource plate_3 was assigned to the liquid handler.
Websocket server started at http://127.0.0.1:2125
File server started at http://127.0.0.1:1341 . Open this URL in your browser.


At this point you should have somethign like this

<div>
<img src="../figs/full_ot2.png" width="600"/>
</div>

Note the difference in **code** to set up the same deck. You will need to take **special care** to search the resources folder for the equipment you need. The easiest way to do this in VS Code is to utilize the **Find in folder** feature on the `pylabrobot/pylabrobot/resources` folder in the file explorer. 

<div>
<img src="../figs/find_in_folder.png" width="600"/>
</div>

Alternatively, if you are familiar, you can use [grep](https://man7.org/linux/man-pages/man1/grep.1.html) or the GitHub search bar online to find the same information. For the following exercises, we will only provide the **deck layout** imports. You will have to find everything else, which will help you practice navigating open-source code.

**Deck Options**

- `STARLetDeck`
- `STARDeck`
- `VantageDeck`
    - `VantageDeck(size=1.3)` - 2.0m variant not supported yet.
- `OTDeck`
- `EVO100Deck`
- `EVO150Deck`
- `EVO200Deck`


Although, we won't grade you for the actual layout of your deck setup (as long as all the requisite pieces of labware are present, here is an important **note** to keep in minde)

---

When designing a protocol, consider both `layout efficiency` and minimizing contamination. Efficient layout reduces pipette travel time, while thoughtful placement minimizes contamination risks. Place tip racks at the back, stocks in the middle, and plates at the front to allow for `linear movement`. For example, the arm grabs a **tip from slot 7**, aspirates **stock from slot 4**, and dispenses in **slot 1**. Reducing travel time decreases protocol runtime.

Minimizing contamination is akin to sterile technique in manual lab work. In automation, prioritize placement **from clean to contaminated**:

`clean tips -> fresh media -> bacteria -> waste -> bleach`. Tips should move from **most sterile to least sterile**, and cleaned tips should return to their original position. Avoid allowing contaminated tips to fly over sterile items. For example, in PCR, tips holding primers should not pass over sterile water to prevent downstream contamination.

---

### Exercises
1. Using an OT2 deck, set up a deck layout that contains:
    - 1 Eppendorf 2mL Safelock Snapcap Acrylic 24-count tube rack
    - 1 Nest 15 mL 15-count conical tube rack
    - 1 12x15 mL reservoir
    - 3 96-well 360 uL plates

2. Using a Hamilton STAR deck, create a plate staircase.
    - A plate staircase shoudl have six plate carriers.
    - There should be a gap of at least 1 rail between each carrier.
    - The leftmost plate staircase should have 0 plates.
    - The rightmorst plate staircase should have 5 plates.
    - Progressively more plates should be added as you move left to right, creating a "staircase" shape
    - For the STAR deck, start at rail 7.

3. Using an OT2 deck, set up a deck layout that contains:
    - A PCR Plate Adapter in slot 10
    - A PCR Plate Adapter + a PCR plate in slot 11
    - For slots 1-9, design an alternating grid, where:
        - There are the highest possible number of wells possible (given the equipment available in the pylabrobot/resources/opentrons folder) on the X (slots 1,3,5,7,9)
        - There are the fewest number of wells possible (given the available equipemnt in pylabrobot/resources/opentrons folder) on the O (slots 2,4,6,8). Hint - a reservoir can have 1 big well sometimes.

4. Using a deck of your choice, set up a deck layout for performing a serial dilution of food colorings. Make sure you have enough tips, solution, and the plates/wells are big enough to handle the steps of this protocol.
    - You will start with stock solutions of at least 100 mL each of 1M red, blue, and yellow dye
    - You will be doing 8 serial dilutions of each color, in triplicate
    - Your maximum transfer volume at one point (max volume in one tip) will be 500 uL
    - The maximum volume in a serial dilution well at any point will be 1 mL
    - Don't forget you need diluent/solvent.

5. Write a 1 sentence justification per equipment choise as why it will satisfy the requirements in exercise 4. Comment on whether your deck layout is organized from most sterile to least sterile.

In [69]:
# Exercise 1
from pylabrobot.resources import (
    opentrons_24_tuberack_eppendorf_2ml_safelock_snapcap_acrylic,
    opentrons_15_tuberack_nest_15ml_conical,
    nest_12_reservoir_15ml,
    corning_96_wellplate_360ul_flat
)

async def exercise_1():
    deck = OTDeck()
    plate_slots = [1, 2, 3]
    deck.assign_child_at_slot(nest_12_reservoir_15ml(name = "reservoir_1"), 4)
    deck.assign_child_at_slot(opentrons_15_tuberack_nest_15ml_conical(name = "conical_tuberack_1"), 5)
    deck.assign_child_at_slot(opentrons_24_tuberack_eppendorf_2ml_safelock_snapcap_acrylic(name = "safelock_acryllic_tuberack_1"), 6)
    for i, plate_slot in enumerate(plate_slots):
        deck.assign_child_at_slot(corning_96_wellplate_360ul_flat(name = f"plate_{i}"), plate_slot)
    return deck
deck = await exercise_1()
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

Setting up the liquid handler.
Resource deck was assigned to the liquid handler.
Resource trash_container was assigned to the liquid handler.
Resource reservoir_1 was assigned to the liquid handler.
Resource conical_tuberack_1 was assigned to the liquid handler.
Resource safelock_acryllic_tuberack_1 was assigned to the liquid handler.
Resource plate_0 was assigned to the liquid handler.
Resource plate_1 was assigned to the liquid handler.
Resource plate_2 was assigned to the liquid handler.
Websocket server started at http://127.0.0.1:2153
File server started at http://127.0.0.1:1369 . Open this URL in your browser.


In [ ]:
# Exercise 2
from pylabrobot.resources import (
    PLT_CAR_L5AC_A00,
    Cor_96_wellplate_360ul_Fb,
    STARDeck
)

async def exercise_2():
    deck = STARDeck()
    plate_num = 0
    for i, n in enumerate(range(7, 43, 7)):
        plate_carrier = PLT_CAR_L5AC_A00(name = f"plate_carrier_{i}")
        for j in range(i):
            plate_carrier[j] = Cor_96_wellplate_360ul_Fb(name=f"plate_{plate_num}")
            plate_num += 1
        deck.assign_child_resource(plate_carrier, rails = n)
    return deck
deck = await exercise_2()
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

Setting up the liquid handler.
Resource deck was assigned to the liquid handler.
Resource trash was assigned to the liquid handler.
Resource trash_core96 was assigned to the liquid handler.
Resource waste_block was assigned to the liquid handler.
Resource plate_carrier_0 was assigned to the liquid handler.
Resource plate_carrier_1 was assigned to the liquid handler.
Resource plate_carrier_2 was assigned to the liquid handler.
Resource plate_carrier_3 was assigned to the liquid handler.
Resource plate_carrier_4 was assigned to the liquid handler.
Resource plate_carrier_5 was assigned to the liquid handler.
Websocket server started at http://127.0.0.1:2152
File server started at http://127.0.0.1:1368 . Open this URL in your browser.


In [67]:
# Exercise 3
from pylabrobot.resources import (
    Opentrons_96_adapter_Vb,
    nest_96_wellplate_100ul_pcr_full_skirt,
    corning_384_wellplate_112ul_flat,
    nest_1_reservoir_290ml
)

async def exercise_3():
    deck = OTDeck()
    adapter_1 = Opentrons_96_adapter_Vb(name="plate_adapter_1")
    adapter_2 = Opentrons_96_adapter_Vb(name="plate_adapter_2")
    skirt_pcr_plate = nest_96_wellplate_100ul_pcr_full_skirt(name="skirted_plate")
    adapter_1.assign_child_resource(skirt_pcr_plate)
    deck.assign_child_at_slot(adapter_1, 11)
    deck.assign_child_at_slot(adapter_2, 10)
    many_wells_slots = [1, 3, 5, 7, 9]
    few_wells_slots =[2, 4, 6, 8]
    for i, many_wells_slot in enumerate(many_wells_slots):
        deck.assign_child_at_slot(corning_384_wellplate_112ul_flat(name = f"384_well_plate_{i}"), many_wells_slot)
    for i, few_wells_slot in enumerate(few_wells_slots):
        deck.assign_child_at_slot(nest_1_reservoir_290ml(name = f"1_well_plate_{i}"), few_wells_slot)
    return deck
deck = await exercise_3()
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

Setting up the liquid handler.
Resource deck was assigned to the liquid handler.
Resource trash_container was assigned to the liquid handler.
Resource plate_adapter_1 was assigned to the liquid handler.
Resource plate_adapter_2 was assigned to the liquid handler.
Resource 384_well_plate_0 was assigned to the liquid handler.
Resource 384_well_plate_1 was assigned to the liquid handler.
Resource 384_well_plate_2 was assigned to the liquid handler.
Resource 384_well_plate_3 was assigned to the liquid handler.
Resource 384_well_plate_4 was assigned to the liquid handler.
Resource 1_well_plate_0 was assigned to the liquid handler.
Resource 1_well_plate_1 was assigned to the liquid handler.
Resource 1_well_plate_2 was assigned to the liquid handler.
Resource 1_well_plate_3 was assigned to the liquid handler.
Websocket server started at http://127.0.0.1:2151
File server started at http://127.0.0.1:1367 . Open this URL in your browser.


In [74]:
# Exercise 4
from pylabrobot.resources import (
    nest_1_reservoir_195ml,
    opentrons_24_tuberack_eppendorf_2ml_safelock_snapcap,
    opentrons_96_tiprack_1000ul
)

async def exercise_4():
    deck = OTDeck()
    dilution_slots = [7, 8, 9]
    for i, dilution_slot in enumerate(dilution_slots):
        deck.assign_child_at_slot(opentrons_24_tuberack_eppendorf_2ml_safelock_snapcap(name = f"serial_dilution_tube_rack_{i}"), dilution_slot)
    tip_spots = [1, 2]
    for i, tip_spot in enumerate(tip_spots):
        deck.assign_child_at_slot(opentrons_96_tiprack_1000ul(name = f"tip_rack_{i}"), tip_spot)
    diluent_reservoir = nest_1_reservoir_195ml(name = "diluent_reservoir")
    deck.assign_child_at_slot(diluent_reservoir, 3)
    colors = ["red", "blue", "yellow"]
    reservoir_slots = [4, 5, 6]
    for color, reservoir_slot in zip(colors, reservoir_slots):
        deck.assign_child_at_slot(nest_1_reservoir_195ml(name = f"{color}_reservoir"), reservoir_slot)
    return deck

deck = await exercise_4()
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

Setting up the liquid handler.
Resource deck was assigned to the liquid handler.
Resource trash_container was assigned to the liquid handler.
Resource serial_dilution_tube_rack_0 was assigned to the liquid handler.
Resource serial_dilution_tube_rack_1 was assigned to the liquid handler.
Resource serial_dilution_tube_rack_2 was assigned to the liquid handler.
Resource tip_rack_0 was assigned to the liquid handler.
Resource tip_rack_1 was assigned to the liquid handler.
Resource diluent_reservoir was assigned to the liquid handler.
Resource red_reservoir was assigned to the liquid handler.
Resource blue_reservoir was assigned to the liquid handler.
Resource yellow_reservoir was assigned to the liquid handler.
Websocket server started at http://127.0.0.1:2156
File server started at http://127.0.0.1:1372 . Open this URL in your browser.


**Exercise 5**

_(insert answer here)_

- 195 mL Reservoir -> Big enough to hold 100 mL of each starting color dye + diluent -> choose 4 reservoirs.
- 8 serial dilutions in triplicate -> at least 24 wells needed per color + max volume of a dilution is 1 mL -> choose 3 separate 24-tube 2 mL racks (could also do deep well plate, etc.)
- Max transfer volume is 500 uL + will need 1-2 tips per dilution -> select 2 x 96 x 1 mL tip boxes
- For sterility, I have organized layout in order from clean tips -> fresh diluent -> color dyes -> serial dilutions -> waste, which is most sterile to least sterile.